In [ ]:
%pip install -q "transformers==4.57.1" "peft==0.17.1" accelerate bitsandbytes datasets huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import getpass
from huggingface_hub import login, whoami, HfApi

hf_token = getpass.getpass("Enter your Hugging Face read token: ")
login(token=hf_token, add_to_git_credential=False)

print("Logged in as:", whoami(token=hf_token)["name"])

# Verify access using this exact account/token
HfApi().model_info(
    "meta-llama/Llama-3.2-1B-Instruct",
    token=hf_token,
)
print("Model access verified.")

Logged in as: Styr0Karam
Model access verified.


In [4]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA unavailable")

2.5.1+cu121
True
NVIDIA GeForce GTX 1650


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    get_peft_model,
    LoraConfig,
    prepare_model_for_kbit_training,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Restart the notebook kernel and select the "
        "Python 3.10 environment with CUDA-enabled PyTorch."
    )

device = torch.device("cuda")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA runtime: {torch.version.cuda}")

print("Loading cached teacher data...")
teacher_data = torch.load(
    "teacher_logits.pt",
    map_location="cpu",
    weights_only=True,
)

student_id = "meta-llama/Llama-3.2-1B-Instruct"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

student_model = AutoModelForCausalLM.from_pretrained(
    student_id,
    quantization_config=quantization_config,
    device_map="auto",
)
student_model.config.use_cache = False
student_model = prepare_model_for_kbit_training(
    student_model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
)
student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()
optimizer = torch.optim.AdamW(
    (parameter for parameter in student_model.parameters() if parameter.requires_grad),
    lr=2e-4,
)

print("Starting Knowledge Distillation...")
student_model.train()
temperature = 2.0

for step, batch in enumerate(teacher_data):
    input_ids = batch["input_ids"].unsqueeze(0).to(device)
    teacher_topk_logits = batch["topk_logits"].unsqueeze(0).to(
        device=device,
        dtype=torch.float16,
    )
    teacher_topk_indices = batch["topk_indices"].unsqueeze(0).to(device)

    student_outputs = student_model(
        input_ids=input_ids,
        use_cache=False,
    )
    student_aligned_logits = torch.gather(
        student_outputs.logits.float(),
        2,
        teacher_topk_indices,
    ).to(torch.float16)
    del student_outputs

    teacher_probs = F.softmax(teacher_topk_logits / temperature, dim=-1)
    student_log_probs = F.log_softmax(
        student_aligned_logits / temperature,
        dim=-1,
    )
    loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean",
    ) * (temperature ** 2)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    if step % 10 == 0:
        print(f"Step {step} | Distillation Loss: {loss.item():.4f}")

print("Distillation complete! You can now save your student model.")
student_model.save_pretrained("./distilled_medical_student")

Using GPU: NVIDIA GeForce GTX 1650
CUDA runtime: 12.1
Loading cached teacher data...
trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689
Starting Knowledge Distillation...


c:\Users\karam\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step 0 | Distillation Loss: 76.4375
Step 10 | Distillation Loss: 56.4375
Step 20 | Distillation Loss: 57.6250
Step 30 | Distillation Loss: 74.5000
Step 40 | Distillation Loss: 55.0938
Step 50 | Distillation Loss: 89.1875
Step 60 | Distillation Loss: 45.2188
Step 70 | Distillation Loss: 50.4062
Step 80 | Distillation Loss: 49.5000
Step 90 | Distillation Loss: 108.1875
Distillation complete! You can now save your student model.


In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

for variable_name in (
    "student_model",
    "optimizer",
    "teacher_data",
    "distilled_model",
    "base_model",
):
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("CUDA is required for inference in this notebook.")

model_id = "meta-llama/Llama-3.2-1B-Instruct"
adapter_path = "./distilled_medical_student"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def make_quant_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=make_quant_config(),
    device_map="auto",
    max_memory={0: "3500MiB", "cpu": "12GiB"},
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = True

distilled_model = PeftModel.from_pretrained(base_model, adapter_path)
distilled_model.eval()
model_device = next(distilled_model.parameters()).device

prompt = (
    "Clinical summary: A 54-year-old male presents with sudden chest tightness "
    "and dyspnea. Initial assessment indicates"
)
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {name: value.to(model_device) for name, value in inputs.items()}

with torch.inference_mode():
    outputs = distilled_model.generate(
        **inputs,
        max_new_tokens=32,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

print("Generated Response:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Generated Response:

Clinical summary: A 54-year-old male presents with sudden chest tightness and dyspnea. Initial assessment indicates that the patient is hypoxic (O2 saturation <80%) in the lungs, but does not have obvious cardiac or pulmonary disease on exam. The patient's


In [ ]:
import math
import os
import torch

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

from datasets import load_dataset

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for validation in this notebook.")

val_data = load_dataset(
    "Saminx22/medical_data_for_slm",
    split="validation[:50]",
)

validation_device = next(distilled_model.parameters()).device
distilled_model.eval()
total_loss = 0.0
total_batches = 0

with torch.inference_mode():
    for item in val_data:
        text = item["text"][:512]
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256,
        )
        inputs = {
            name: value.to(validation_device)
            for name, value in inputs.items()
        }
        inputs["labels"] = inputs["input_ids"].clone()

        outputs = distilled_model(**inputs, use_cache=False)
        total_loss += outputs.loss.item()
        total_batches += 1

        del inputs, outputs

avg_val_loss = total_loss / total_batches
perplexity = math.exp(avg_val_loss)

print(f"Validation Loss: {avg_val_loss:.4f}")
print(f"Validation Perplexity (PPL): {perplexity:.2f}")

Validation Loss: 3.1154
Validation Perplexity (PPL): 22.54
